# 02 — Build the flood model


**Step 1 — load & inspect.** Load the cached inputs `x`, per-frame lead times `t`, and labels `y` for one sample, and look at the **raw tensors** (shapes, dtypes, values) — no plots yet.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# repo-root config (single source of truth)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import CACHE_DIR, BANDS, N_BAND, T_FRAMES

# every cached sample + its train/val/test split
manifest = pd.read_parquet(CACHE_DIR / "manifest.parquet")
print(f"{len(manifest)} samples | splits {manifest['split'].value_counts().to_dict()}")

# load ONE sample's raw arrays straight off the cache
date = manifest["label_day"].iloc[0].strftime("%Y%m%d")
x = np.load(CACHE_DIR / f"{date}_x.npy")     # GOES   (T, band, H, W)  float16
t = np.load(CACHE_DIR / f"{date}_t.npy")     # lead   (T,)  hours before day D
y = np.load(CACHE_DIR / f"{date}_y.npy")     # floods (R, C)  0/1

print(f"\nsample {date}  (bands {BANDS}, T={T_FRAMES})")
print(f"  x {x.shape} {x.dtype}  min {x.min():.2f} max {x.max():.2f} mean {x.mean():.3f}")
print(f"  t {t.shape} {t.dtype}  = {t}")
print(f"  y {y.shape} {y.dtype}  flooded {int(y.sum())} of {y.size} cells")

# the actual numbers, not pictures - a CENTER patch (image corners are off-Earth = 0)
cy, cx = x.shape[2] // 2, x.shape[3] // 2
print(f"\nx[frame 0, band 0]  5x5 center patch @ ({cy},{cx}) (normalized):")
print(np.round(x[0, 0, cy:cy + 5, cx:cx + 5].astype(np.float32), 2))
print("\nt (hours before day D, per frame):", t)
fl = np.argwhere(y == 1)
print(f"\ny flooded-cell coords [row, col] (first 12 of {len(fl)}):")
print(fl[:12])

## 2. Pixel → cell index (for the geographic pool)

The geographic pool needs to know **which 50 km cell each ABI pixel belongs to**. We project every pixel of the GOES fixed grid into CONUS Albers and integer-divide it onto the cell lattice (same `grid_transform()` the labels use), giving a `(1500, 2500)` array of flat cell ids (`-1` = off-grid). The model later subsamples this to the encoder's resolution and averages each cell's pixels.

In [ ]:
import netCDF4
import pyproj

from config import DATA_DIR, build_grid_cells, grid_transform

# target: the 50 km cell grid (R, C) + which cells are land
cells, GRID_R, GRID_C, land_mask = build_grid_cells()
GX0, GY0, GSTEP, _r, _c = grid_transform()
assert (_r, _c) == (GRID_R, GRID_C)

# ABI pixel centres from the GOES fixed-grid projection (any 2019 file works)
ref = sorted(DATA_DIR.glob("*/2019/*/*/*.nc"))[0]
with netCDF4.Dataset(ref) as nc:
    proj = nc["goes_imager_projection"]
    geos = pyproj.CRS.from_cf({k: proj.getncattr(k) for k in proj.ncattrs()})
    
    sat_h = float(proj.perspective_point_height)
    xc = nc["x"][:].astype(np.float64) * sat_h      # column scan-angle -> metres
    yc = nc["y"][:].astype(np.float64) * sat_h      # row scan-angle    -> metres
XX, YY = np.meshgrid(xc, yc)                          # (1500, 2500) each

# project every ABI pixel to Albers, then integer-divide onto the cell lattice
to_albers = pyproj.Transformer.from_crs(geos, 5070, always_xy=True)
ax, ay = to_albers.transform(XX.ravel(), YY.ravel())
ax = np.asarray(ax).reshape(XX.shape)
ay = np.asarray(ay).reshape(XX.shape)
col = np.floor((ax - GX0) / GSTEP)
row = (GRID_R - 1) - np.floor((ay - GY0) / GSTEP)
ok = (np.isfinite(ax) & np.isfinite(ay)
      & (col >= 0) & (col < GRID_C) & (row >= 0) & (row < GRID_R))
pix2cell = np.full(ax.shape, -1, np.int32)            # (1500, 2500), -1 = off-grid
pix2cell[ok] = (row[ok] * GRID_C + col[ok]).astype(np.int32)

# ---- preview ----
counts = np.bincount(pix2cell[pix2cell >= 0], minlength=GRID_R * GRID_C)
nz = counts[counts > 0]
print(f"pix2cell {pix2cell.shape} {pix2cell.dtype}  (pixel -> flat cell id, -1 = off-grid)")
print(f"  valid pixels : {(pix2cell >= 0).sum():,} of {pix2cell.size:,} "
      f"({(pix2cell >= 0).mean():.0%})")
print(f"  cells covered: {len(nz)} of {GRID_R * GRID_C}  "
      f"({int(land_mask.sum())} land cells)")
print(f"  pixels/cell  : min {nz.min()}, median {int(np.median(nz))}, max {nz.max()}")
print(f"  land cells with 0 pixels: {int((counts[land_mask.ravel()] == 0).sum())}  (want 0)")
print("\npix2cell[750:755, 1250:1255]  (center patch, flat cell ids):")
print(pix2cell[750:755, 1250:1255])
_id = int(pix2cell[750, 1250])
print(f"\nflat id {_id}  ->  (row {_id // GRID_C}, col {_id % GRID_C})")

## 3. Per-frame encoder

A small CNN applied to **each of the 6 frames with shared weights**. `conv_block` = two `Conv3x3 -> GroupNorm -> ReLU`; two MaxPools downsample **÷4** (`1500×2500 -> 375×625`) while lifting `N_CH(=7) -> 64` channels. The downsample factor (`stride=4`) lives **here in the model**, not config — the grid pool keeps full resolution otherwise. Same 'eye' looks at every frame.

In [ ]:
import torch
import torch.nn as nn

from config import N_CH, IMG_H, IMG_W


def conv_block(cin, cout):
    """two 3x3 convs, each GroupNorm(8) + ReLU."""
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
    )


class Encoder(nn.Module):
    """Per-frame CNN: (B, N_CH, 1500, 2500) -> (B, 64, 375, 625), downsampled /4."""
    def __init__(self, cin=N_CH):
        super().__init__()
        self.b1 = conv_block(cin, 32)
        self.b2 = conv_block(32, 64)
        self.b3 = conv_block(64, 64)
        self.pool = nn.MaxPool2d(2)
        self.out_ch = 64
        self.stride = 4                  # total spatial downsample (model knob)

    def forward(self, x):
        x = self.pool(self.b1(x))        # /2 -> 750x1250
        x = self.pool(self.b2(x))        # /4 -> 375x625
        x = self.b3(x)                   # refine, no downsample
        return x


# shape smoke test (one dummy frame, on GPU if available)
dev = "cuda" if torch.cuda.is_available() else "cpu"
enc = Encoder().to(dev)
dummy = torch.zeros(1, N_CH, IMG_H, IMG_W, device=dev)
with torch.no_grad():
    out = enc(dummy)
print(f"encoder on {dev}:  {tuple(dummy.shape)}  ->  {tuple(out.shape)}")
print(f"  out channels {enc.out_ch}, stride /{enc.stride}")
print(f"  params: {sum(p.numel() for p in enc.parameters()):,}")

## 4. ConvLSTM (time fusion)

Walks the 6 encoder maps **in order** (frame 0→5), carrying a 2-D spatial hidden state `h` and cell state `c` — this is Keras `ConvLSTM2D`. Each step's gates are a single convolution over `[xₜ, hₜ₋₁]` split into input/forget/output/candidate. Returns the **final hidden map** = the fused summary of how the day evolved. Input `(B, 6, 64, 375, 625)` → output `(B, 64, 375, 625)`.

In [ ]:
class ConvLSTMCell(nn.Module):
    """One ConvLSTM step: gates are a conv over [x, h]."""
    def __init__(self, cin, ch, k=3):
        super().__init__()
        self.ch = ch
        self.conv = nn.Conv2d(cin + ch, 4 * ch, k, padding=k // 2)

    def forward(self, x, h, c):
        i, f, o, g = self.conv(torch.cat([x, h], 1)).chunk(4, 1)
        c = f.sigmoid() * c + i.sigmoid() * g.tanh()   # forget old + write new
        h = o.sigmoid() * c.tanh()                     # exposed hidden state
        return h, c


class ConvLSTM(nn.Module):
    """Walk the T frames in order; return the final hidden feature map."""
    def __init__(self, cin, ch=64, k=3):
        super().__init__()
        self.cell = ConvLSTMCell(cin, ch, k)
        self.ch = ch

    def forward(self, seq):                  # (B, T, cin, H, W)
        B, T, _, H, W = seq.shape
        h = seq.new_zeros(B, self.ch, H, W)
        c = seq.new_zeros(B, self.ch, H, W)
        for t in range(T):                   # chronological 0 -> T-1
            h, c = self.cell(seq[:, t], h, c)
        return h                             # (B, ch, H, W)


# smoke test: 6 encoder-sized frames -> one fused map
clstm = ConvLSTM(cin=enc.out_ch, ch=64).to(dev)
seq = torch.zeros(1, T_FRAMES, enc.out_ch, 375, 625, device=dev)
with torch.no_grad():
    fused = clstm(seq)
print(f"convLSTM on {dev}:  {tuple(seq.shape)}  ->  {tuple(fused.shape)}")
print(f"  params: {sum(p.numel() for p in clstm.parameters()):,}")

## 5. Geographic grid pool (`CellPool`)

Subsample the pixel→cell index to the encoder's resolution (÷4 → 375×625) and **scatter-mean** each cell's feature pixels onto the 50 km grid: `(B, 64, 375, 625) -> (B, 64, 59, 95)`. The index + per-cell pixel counts are registered as buffers (so they ride to the GPU); off-grid pixels (`-1`) go to a discarded dump bin. This is the geographic regrid — exact, not a resize.

In [ ]:
class CellPool(nn.Module):
    """Scatter-mean encoder pixels into the (R, C) cells via a fixed index."""
    def __init__(self, pix2cell_sub, grid_r, grid_c):
        super().__init__()
        n = grid_r * grid_c
        idx = torch.from_numpy(pix2cell_sub.reshape(-1).astype(np.int64)).clone()
        idx[idx < 0] = n                              # off-grid -> dump bin
        counts = torch.zeros(n + 1).scatter_add_(
            0, idx, torch.ones(idx.numel()))
        self.register_buffer("idx", idx)
        self.register_buffer("counts", counts.clamp_min(1.0))
        self.n, self.grid_r, self.grid_c = n, grid_r, grid_c

    def forward(self, x):                             # (B, C, h, w)
        B, C, h, w = x.shape
        flat = x.reshape(B, C, h * w)
        out = x.new_zeros(B, C, self.n + 1)
        out.scatter_add_(2, self.idx.view(1, 1, -1).expand(B, C, -1), flat)
        out = out / self.counts.view(1, 1, -1)        # -> per-cell mean
        return out[:, :, :self.n].reshape(B, C, self.grid_r, self.grid_c)


# subsample the full-res index to the encoder grid (block centres) and build pool
S = enc.stride
sub = pix2cell[S // 2::S, S // 2::S][:IMG_H // S, :IMG_W // S]   # (375, 625)
pool = CellPool(sub, GRID_R, GRID_C).to(dev)

with torch.no_grad():
    cellmap = pool(fused)
    chk = pool(torch.ones(1, 1, *sub.shape, device=dev))     # pooled ones
land = torch.from_numpy(land_mask).to(dev)
print(f"cellPool on {dev}:  {tuple(fused.shape)}  ->  {tuple(cellmap.shape)}")
print(f"  sub index {sub.shape}  | sanity: pooled-ones over land = "
      f"{chk[0, 0][land].mean():.3f} (want 1.000)")

## 6. Assemble — `FloodConvLSTM`

Wire the four stages end to end:

```
(B,6,7,1500,2500)
  encoder (shared weights, per frame)  -> (B,6,64,375,625)
  convLSTM (fuse time)                 -> (B,64,375,625)
  cellPool (regrid)                    -> (B,64,59,95)
  head: conv_block -> 1x1 conv         -> (B,59,95) logits
```

Sigmoid of the logits = P(flood) per cell. Full forward on a dummy batch.

In [ ]:
class FloodConvLSTM(nn.Module):
    """encoder (per frame) -> ConvLSTM (time) -> CellPool (grid) -> head."""
    def __init__(self, pix2cell_sub, grid_r, grid_c, cin=N_CH):
        super().__init__()
        self.encoder = Encoder(cin)
        self.convlstm = ConvLSTM(self.encoder.out_ch, ch=64)
        self.pool = CellPool(pix2cell_sub, grid_r, grid_c)
        self.head = nn.Sequential(conv_block(64, 64), nn.Conv2d(64, 1, 1))

    def forward(self, x):                        # (B, T, C, H, W)
        B, T = x.shape[:2]
        feats = self.encoder(x.flatten(0, 1))    # shared weights over frames
        feats = feats.unflatten(0, (B, T))       # (B, T, 64, h, w)
        fused = self.convlstm(feats)             # (B, 64, h, w)
        cells = self.pool(fused)                 # (B, 64, R, C)
        return self.head(cells).squeeze(1)       # (B, R, C) logits


model = FloodConvLSTM(sub, GRID_R, GRID_C).to(dev)
x_dummy = torch.zeros(1, T_FRAMES, N_CH, IMG_H, IMG_W, device=dev)
with torch.no_grad():
    logits = model(x_dummy)
print(f"FloodConvLSTM on {dev}:  {tuple(x_dummy.shape)}  ->  {tuple(logits.shape)}")
print(f"  total params: {sum(p.numel() for p in model.parameters()):,}")

## 7. Dataset

`FloodCache` loads one cached sample and assembles the model input: the 6-band GOES sequence `_x` plus, when `USE_TIME`, the per-frame lead time `_t` appended as a **7th channel** (`lead/24`, broadcast over H×W) → `(T, 7, H, W)`. Paired with the label `_y (59,95)`. Splits come straight from the manifest. Returns float16 x (we cast to float on the GPU) to keep loader memory down.

In [ ]:
from torch.utils.data import Dataset, DataLoader

from config import USE_TIME


class FloodCache(Dataset):
    """Cached sample -> (x (T,7,H,W) f16, y (R,C) f32); appends lead-time channel."""
    def __init__(self, split):
        m = pd.read_parquet(CACHE_DIR / "manifest.parquet")
        self.dates = [d.strftime("%Y%m%d")
                      for d in m.loc[m.split == split, "label_day"]]

    def __len__(self):
        return len(self.dates)

    def __getitem__(self, i):
        d = self.dates[i]
        x = np.load(CACHE_DIR / f"{d}_x.npy")                   # (T, 6, H, W) f16
        y = np.load(CACHE_DIR / f"{d}_y.npy").astype(np.float32)  # (R, C)
        if USE_TIME:
            t = np.load(CACHE_DIR / f"{d}_t.npy")               # (T,) lead hours
            T, _, H, W = x.shape
            lead = np.empty((T, 1, H, W), np.float16)
            lead[:] = (t / 24.0).astype(np.float16).reshape(T, 1, 1, 1)
            x = np.concatenate([x, lead], axis=1)               # (T, 7, H, W)
        return torch.from_numpy(x), torch.from_numpy(y)


train_ds, val_ds, test_ds = FloodCache("train"), FloodCache("val"), FloodCache("test")
print(f"datasets: train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

xb, yb = train_ds[0]
print(f"one sample: x {tuple(xb.shape)} {xb.dtype} | y {tuple(yb.shape)} {yb.dtype}")
print(f"  per-channel mean: {xb.float().mean((0, 2, 3)).round(decimals=2).tolist()}")
print(f"  channel 6 (lead/24) per frame: {xb[:, 6, 0, 0].float().round(decimals=2).tolist()}")

# Dataset -> model compatibility
with torch.no_grad():
    out = model(xb.unsqueeze(0).float().to(dev))
print(f"  model(x[None]) -> {tuple(out.shape)}  {out.dtype}")

## 8. Loss — soft Tversky  *(this is the backprop loss)*

**Differentiable**: it runs on `sigmoid(logits)` probabilities (not a hard 0/1 threshold), so gradients flow and this is exactly what the optimizer minimizes.

$$\mathcal{L} = 1 - \frac{TP}{TP + \alpha\,FP + \beta\,FN}$$

computed **only on land cells**, with hard 0/1 targets and α=0.7 > β=0.3 so false positives are penalized harder than false negatives (discourages over-predicting flood everywhere). The hard binary metrics (F1/IoU) come later and are **eval-only** — never backpropped.

In [ ]:
from config import TVERSKY_ALPHA, TVERSKY_BETA

land_t = torch.from_numpy(land_mask).to(dev)        # (R, C) bool, score here only


def tversky_loss(logits, y, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, smooth=1.0):
    """Soft Tversky over land cells -> the differentiable training (backprop) loss."""
    p = torch.sigmoid(logits)[:, land_t]            # (B, n_land) probabilities
    yt = y[:, land_t]
    tp = (p * yt).sum(1)
    fp = (p * (1 - yt)).sum(1)
    fn = ((1 - p) * yt).sum(1)
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return (1 - tversky).mean()


# smoke test: real sample -> loss -> backward (proves gradients flow)
xb, yb = train_ds[0]
xb = xb.unsqueeze(0).float().to(dev)
yb = yb.unsqueeze(0).to(dev)
model.zero_grad()
loss = tversky_loss(model(xb), yb)
loss.backward()
gnorm = torch.cat([prm.grad.flatten() for prm in model.parameters()
                   if prm.grad is not None]).norm().item()
print(f"tversky loss = {loss.item():.4f}")
print(f"grad norm    = {gnorm:.3e}   (> 0 -> backprop works, this loss trains the model)")
model.zero_grad()

## 9. Train — DDP script (both GPUs, fastest)

Training runs in **`train_ddp.py`** via `torchrun` — true **DDP + bf16 across both GPUs** (the fast mode), as separate processes (so it's independent of this kernel's CUDA state). It saves the best-val checkpoint to `CKPT_DIR`; we load it below for metrics + predictions. The model/data/loss live in **`floodnet.py`**, imported by both the script and this notebook.

Run the cell to launch (~few min/epoch). Output streams here. Tip: `SMOKE=1` env → quick 1-epoch subset.

In [ ]:
import subprocess
import sys
from pathlib import Path

# locate notebooks/model/ (the kernel's cwd is often the repo root, not here)
_root = Path.cwd()
while not (_root / "config.py").exists() and _root != _root.parent:
    _root = _root.parent
_model_dir = _root / "notebooks" / "model"

# train on BOTH GPUs via DDP in a separate process group (torchrun).
subprocess.run(
    [sys.executable, "-m", "torch.distributed.run",
     "--nproc_per_node=2", "train_ddp.py"],
    cwd=str(_model_dir), check=True,
)

## 10. Metrics (from the trained checkpoint)

Load the best checkpoint into a fresh `FloodConvLSTM` and score the **test** split — threshold-free **PR-AUC / ROC-AUC** plus a per-cell classification report at the 0.5 threshold. (Runs single-GPU in this kernel.)

In [ ]:
import sys
from pathlib import Path

import torch
from sklearn.metrics import (average_precision_score, classification_report,
                             roc_auc_score)
from torch.utils.data import DataLoader

# floodnet.py lives in notebooks/model/ (the kernel's cwd may be the repo root)
_md = Path.cwd()
while not (_md / "config.py").exists() and _md != _md.parent:
    _md = _md.parent
sys.path.insert(0, str(_md / "notebooks" / "model"))

from config import CKPT_DIR, PRED_THRESHOLD
from floodnet import FloodCache, FloodConvLSTM, build_pix2cell, pool_sub

p2c, GR, GC, land = build_pix2cell()
maskb = torch.from_numpy(land)
ev_dev = "cuda" if torch.cuda.is_available() else "cpu"
net = FloodConvLSTM(pool_sub(p2c), GR, GC).to(ev_dev).eval()
net.load_state_dict(torch.load(CKPT_DIR / "floodconvlstm_best.pt", map_location=ev_dev))

test_ds = FloodCache("test")
probs, trues = [], []
with torch.no_grad():
    for xb, yb in DataLoader(test_ds, batch_size=1, num_workers=4):
        pr = torch.sigmoid(net(xb.float().to(ev_dev))).cpu()
        probs.append(pr[:, maskb].flatten())
        trues.append(yb[:, maskb].flatten())
probs = torch.cat(probs).numpy()
trues = torch.cat(trues).numpy().astype(int)

print(f"test cells: {len(trues):,} | positive rate {trues.mean():.3%}\n")
print(f"PR-AUC : {average_precision_score(trues, probs):.4f}")
print(f"ROC-AUC: {roc_auc_score(trues, probs):.4f}\n")
print(classification_report(trues, probs > PRED_THRESHOLD,
                            target_names=["no flood", "flood"], digits=3))

## 11. Predictions vs truth

Predicted flood vs actual flooded cells for a few random test days. Overlay: **red = missed truth, blue = false alarm, purple = correct hit**, grey = ocean; per-day F1 in the title.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score

land_np = land.astype(bool)
rng = np.random.default_rng(0)
pick = rng.choice(len(test_ds), size=min(8, len(test_ds)), replace=False)
ncol = 4
nrow = int(np.ceil(len(pick) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.6 * nrow),
                         constrained_layout=True)
for ax, i in zip(np.atleast_1d(axes).flat, pick):
    xb, yb = test_ds[i]
    with torch.no_grad():
        prob = torch.sigmoid(net(xb[None].float().to(ev_dev)))[0].cpu().numpy()
    truth = yb.numpy()
    pred = (prob > PRED_THRESHOLD).astype(float)
    f1 = f1_score(truth[land_np].ravel(), pred[land_np].ravel(), zero_division=0)
    rgb = np.ones((*truth.shape, 3))                       # white = correct no-flood
    rgb[(truth > 0) & (pred == 0)] = [1, 0, 0]             # red   = missed
    rgb[(pred > 0) & (truth == 0)] = [0, 0, 1]             # blue  = false alarm
    rgb[(pred > 0) & (truth > 0)] = [0.6, 0, 0.6]          # purple= hit
    rgb[~land_np] = 0.9                                    # grey  = ocean
    ax.imshow(rgb)
    ax.set_title(f"{test_ds.dates[i]}  F1={f1:.2f}", fontsize=8)
    ax.axis("off")
for ax in np.atleast_1d(axes).flat[len(pick):]:
    ax.axis("off")
fig.suptitle("test predictions — red=missed, blue=false alarm, purple=hit", fontsize=11)

## 12. Five test samples on the map

Ground truth vs model prediction for 5 random test days, drawn on the CONUS map (state boundaries + the 50 km cells, equal-area Albers). Each flagged cell is coloured **purple = hit, red = missed flood, blue = false alarm**; per-day F1 in the title.

In [ ]:
import geopandas as gpd
from sklearn.metrics import f1_score

from config import STATES_GEOJSON

# CONUS basemap + the 50 km cells as polygons (Albers, equal-area)
cells_gdf, _, _, _ = build_grid_cells()
cells_gdf = cells_gdf.to_crs(5070)
basemap = gpd.read_file(STATES_GEOJSON)
basemap = basemap[~basemap["name"].isin({"Alaska", "Hawaii", "Puerto Rico"})].to_crs(5070)
land_np = land.astype(bool)
rr, cc = cells_gdf["R"].to_numpy(), cells_gdf["C"].to_numpy()
colors = {"hit": "#7b2cbf", "miss": "#e63946", "false": "#1d6fb8"}

rng = np.random.default_rng(1)
pick = rng.choice(len(test_ds), 5, replace=False)
fig, axes = plt.subplots(1, 5, figsize=(3.8 * 5, 3.4), constrained_layout=True)
for ax, i in zip(axes, pick):
    xb, yb = test_ds[i]
    with torch.no_grad():
        prob = torch.sigmoid(net(xb[None].float().to(ev_dev)))[0].cpu().numpy()
    truth = yb.numpy().astype(bool)
    pred = prob > PRED_THRESHOLD
    tv, pv = truth[rr, cc], pred[rr, cc]
    g = cells_gdf.assign(cat=np.where(tv & pv, "hit",
                                      np.where(tv, "miss", np.where(pv, "false", "none"))))
    f1 = f1_score(truth[land_np].ravel(), pred[land_np].ravel(), zero_division=0)
    basemap.boundary.plot(ax=ax, color="0.75", linewidth=0.4)
    for cat, col in colors.items():
        sel = g[g["cat"] == cat]
        if len(sel):
            sel.plot(ax=ax, color=col, alpha=0.8, edgecolor="none")
    ax.set_title(f"{test_ds.dates[i]}   F1={f1:.2f}", fontsize=9)
    ax.set_axis_off()
handles = [plt.matplotlib.patches.Patch(color=colors[k], label=v)
           for k, v in [("hit", "hit"), ("miss", "missed flood"),
                        ("false", "false alarm")]]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)
fig.suptitle("5 test samples — ground truth vs model prediction (CONUS)", fontsize=12)